# 01 — Data Loading

Studio Proyek 2 · Klasifikasi 5 jenis sampah (IF25-40401)

Notebook ini berisi langkah-langkah memuat dataset sampai siap dipakai untuk pelatihan:

1. Setup & konfigurasi
2. Cek struktur folder dataset
3. Inventaris gambar per kelas
4. Cek integritas gambar (file rusak, ukuran, mode warna)
5. Lihat contoh gambar
6. Split train / validasi lokal (`my_train/`, `my_val/`)
7. Transform (augmentasi train, transform validasi)
8. `ImageFolder` + `DataLoader`
9. Cek satu batch
10. Bobot kelas (untuk ketidakseimbangan kelas)

## 1. Setup & konfigurasi

In [ ]:
import os
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Notebook ada di notebooks/, sedangkan train_baseline.py dan submission/ ada di root repo
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "train"  # folder train/ dari dataset (Google Drive)
EXPECTED_CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

SEED = 42
VAL_FRAC = 0.15
BATCH_SIZE = 32
NUM_WORKERS = 2  # set 0 kalau DataLoader error / macet di Windows

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("root  :", ROOT)
print("device:", device)

## 2. Cek struktur folder

Dataset harus berstruktur `train/<nama_kelas>/*.jpg` dengan 5 folder kelas.
Urutan alfabetis folder = indeks kelas (sama dengan `ImageFolder` dan `evaluate.py`).

In [ ]:
assert DATA_DIR.is_dir(), (
    f"Folder {DATA_DIR} tidak ditemukan. Unduh dataset dari Google Drive "
    "(lihat spesifikasi tugas) lalu ekstrak sehingga ada train/<nama_kelas>/..."
)

found = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print("folder kelas ditemukan:", found)
assert found == EXPECTED_CLASSES, f"Folder kelas tidak sesuai.\n  diharapkan: {EXPECTED_CLASSES}\n  ditemukan : {found}"

for i, c in enumerate(EXPECTED_CLASSES):
    print(f"  {i} -> {c}")

## 3. Inventaris gambar per kelas

In [ ]:
files_by_class = {
    c: sorted(p for p in (DATA_DIR / c).iterdir() if p.is_file()) for c in EXPECTED_CLASSES
}
counts = {c: len(fs) for c, fs in files_by_class.items()}
total = sum(counts.values())

for c, n in counts.items():
    print(f"{c:<15} {n:>5}  ({n / total:.1%})")
print(f"{'TOTAL':<15} {total:>5}")

ext_counter = Counter(p.suffix.lower() for fs in files_by_class.values() for p in fs)
print("\nekstensi file:", dict(ext_counter))

# make_local_split() di train_baseline.py hanya menyalin *.jpg; ekstensi lain akan terlewat
other_ext = {e: n for e, n in ext_counter.items() if e != ".jpg"}
if other_ext:
    print("PERINGATAN: ada file non-.jpg, tidak ikut ter-split oleh make_local_split():", other_ext)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(list(counts), list(counts.values()))
ax.set_ylabel("jumlah gambar")
ax.set_title("Distribusi kelas (train)")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 4. Cek integritas gambar

Buka setiap gambar untuk mendeteksi file rusak, lalu catat ukuran dan mode warna.
Gambar non-RGB (mis. `P`, `L`, `RGBA`) aman: `ImageFolder` dan `evaluate.py` sama-sama memanggil `.convert("RGB")`.

In [ ]:
corrupt, sizes, modes = [], [], Counter()
for c, fs in files_by_class.items():
    for p in fs:
        try:
            with Image.open(p) as im:
                im.verify()  # verify() membuat objek tidak bisa dipakai lagi, jadi buka ulang
            with Image.open(p) as im:
                sizes.append(im.size)
                modes[im.mode] += 1
        except Exception as e:
            corrupt.append((p, repr(e)))

print(f"gambar terbaca : {len(sizes)} / {total}")
print(f"file rusak     : {len(corrupt)}")
for p, err in corrupt:
    print("  ", p.relative_to(ROOT), "->", err)
print("mode warna     :", dict(modes))

ws, hs = np.array([s[0] for s in sizes]), np.array([s[1] for s in sizes])
print(f"lebar  min/median/max: {ws.min()} / {int(np.median(ws))} / {ws.max()}")
print(f"tinggi min/median/max: {hs.min()} / {int(np.median(hs))} / {hs.max()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(ws, bins=30)
axes[0].set_title("Lebar (px)")
axes[1].hist(hs, bins=30)
axes[1].set_title("Tinggi (px)")
plt.tight_layout()
plt.show()

> Kalau ada file rusak: pindahkan/hapus manual dari `train/` **sebelum** langkah split (langkah 6),
> karena split menyalin semua file apa adanya.

## 5. Contoh gambar per kelas

In [ ]:
N_SHOW = 5
rng = random.Random(SEED)
fig, axes = plt.subplots(len(EXPECTED_CLASSES), N_SHOW, figsize=(2.4 * N_SHOW, 2.4 * len(EXPECTED_CLASSES)))
for r, c in enumerate(EXPECTED_CLASSES):
    for col, p in enumerate(rng.sample(files_by_class[c], N_SHOW)):
        ax = axes[r, col]
        with Image.open(p) as im:
            ax.imshow(im.convert("RGB"))
        ax.axis("off")
        if col == 0:
            ax.set_title(c, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Split train / validasi lokal

Validation set asli disimpan dosen dan tidak pernah kita lihat, jadi kita potong sendiri **15% per kelas**
dari `train/` menjadi `my_val/` (sisanya `my_train/`). Memakai `make_local_split()` dari `train_baseline.py`
supaya hasil split identik dengan baseline dan langsung bisa diuji dengan
`python evaluate.py --submission ./submission --data ./my_val`.

Kalau `my_val/` sudah ada, split **tidak diulang** (konsisten antar-eksperimen). Hapus `my_train/` dan `my_val/` bila ingin split ulang.

In [ ]:
from train_baseline import make_local_split

tr_dir, va_dir = make_local_split(DATA_DIR, ".", VAL_FRAC, SEED)
tr_dir, va_dir = Path(tr_dir), Path(va_dir)


def count_split(d):
    return {c: len([p for p in (d / c).iterdir() if p.is_file()]) for c in EXPECTED_CLASSES}


tr_counts, va_counts = count_split(tr_dir), count_split(va_dir)
print(f"{'kelas':<15} {'train':>6} {'val':>6}")
for c in EXPECTED_CLASSES:
    print(f"{c:<15} {tr_counts[c]:>6} {va_counts[c]:>6}")
print(f"{'TOTAL':<15} {sum(tr_counts.values()):>6} {sum(va_counts.values()):>6}")

# Pastikan tidak ada gambar yang bocor di kedua split
tr_names = {(c, p.name) for c in EXPECTED_CLASSES for p in (tr_dir / c).iterdir()}
va_names = {(c, p.name) for c in EXPECTED_CLASSES for p in (va_dir / c).iterdir()}
assert not (tr_names & va_names), "Ada gambar yang sama di my_train dan my_val!"
print("\nOK: tidak ada overlap antara my_train dan my_val")

## 7. Transform

- **Train:** augmentasi (`RandomResizedCrop`, flip, `ColorJitter`) + normalisasi ImageNet — sama dengan baseline.
- **Validasi:** memakai `get_transform()` dari `submission/model.py`. Ini wajib sama persis dengan transform yang dipakai
  `evaluate.py`, jadi jangan didefinisikan ulang di sini.

In [ ]:
from submission.model import get_transform

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = get_transform()


def denorm(x):
    """Tensor [3,H,W] ternormalisasi -> array [H,W,3] di rentang 0..1 untuk imshow."""
    x = x.cpu() * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


# Pratinjau augmentasi: 1 gambar asli + 5 hasil augmentasi acak
sample_path = files_by_class[EXPECTED_CLASSES[0]][0]
with Image.open(sample_path) as im:
    sample = im.convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
axes[0].imshow(sample)
axes[0].set_title("asli")
for ax in axes[1:]:
    ax.imshow(denorm(train_tf(sample)))
    ax.set_title("augmentasi")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. `ImageFolder` + `DataLoader`

In [ ]:
train_ds = datasets.ImageFolder(tr_dir, train_tf)
val_ds = datasets.ImageFolder(va_dir, val_tf)

# Urutan kelas harus sama dengan kontrak submission (alfabetis)
assert train_ds.classes == EXPECTED_CLASSES == val_ds.classes
print("class_to_idx:", train_ds.class_to_idx)
print(f"train: {len(train_ds)} gambar | val: {len(val_ds)} gambar")

train_dl = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device == "cuda"), persistent_workers=NUM_WORKERS > 0,
)
val_dl = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device == "cuda"), persistent_workers=NUM_WORKERS > 0,
)
print(f"batch/epoch: train {len(train_dl)} | val {len(val_dl)}")

## 9. Cek satu batch

In [ ]:
x, y = next(iter(train_dl))
print("x:", tuple(x.shape), x.dtype, f"| min {x.min():.2f} max {x.max():.2f}")
print("y:", tuple(y.shape), y.dtype, "| label unik:", y.unique().tolist())

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, img, label in zip(axes.flat, x, y):
    ax.imshow(denorm(img))
    ax.set_title(EXPECTED_CLASSES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Bobot kelas

Jumlah gambar per kelas tidak seimbang (mis. `kertas_kardus` ≈ 395 vs `saset_kemasan` ≈ 760), sedangkan metrik
penilaian adalah **Macro-F1** (semua kelas berbobot sama). Hitung bobot invers-frekuensi supaya bisa dipakai di loss
(`CrossEntropyLoss(weight=...)`) atau sebagai opsi `WeightedRandomSampler` saat eksperimen.

In [ ]:
train_counts = Counter(train_ds.targets)
n_train, n_cls = len(train_ds), len(EXPECTED_CLASSES)
class_weights = torch.tensor(
    [n_train / (n_cls * train_counts[i]) for i in range(n_cls)], dtype=torch.float
)
for i, c in enumerate(EXPECTED_CLASSES):
    print(f"{c:<15} n={train_counts[i]:>4}  weight={class_weights[i]:.3f}")

## Ringkasan

Objek yang siap dipakai di notebook pelatihan berikutnya:

| Objek | Isi |
| --- | --- |
| `train_ds`, `val_ds` | `ImageFolder` untuk `my_train/` dan `my_val/` |
| `train_dl`, `val_dl` | `DataLoader` (batch 32) |
| `train_tf`, `val_tf` | transform train (augmentasi) dan validasi (`get_transform()`) |
| `class_weights` | bobot kelas untuk loss berbobot |
| `EXPECTED_CLASSES` | urutan kelas 0–4 sesuai kontrak submission |